In [1]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(split_name: str, feature_name: str) -> pd.DataFrame:
    path = f"datasets/{split_name}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(split_name: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(split_name, feature_names[0])
    for name in feature_names[1:]:
        extra = load_split(split_name, name)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    dtrain = lgb.Dataset(train_df[feat_cols], label=train_df["target"])
    dval = lgb.Dataset(val_df[feat_cols], label=val_df["target"], reference=dtrain)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 63,  # ~2^depth for depth=6
        "max_depth": 6,
        "verbosity": -1,
        "seed": 42,
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1),
    ]

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=callbacks,
    )

    preds = model.predict(val_df[feat_cols].values)
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split("split1", feature_name)
    val_df = load_split("split2", feature_name)
    score, best_iter = train_and_eval(train_df, val_df)
    print(f"  RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": feature_name, "groups": feature_name, "rmsle": score})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("split1", subset)
    val_df = merge_splits("split2", subset)
    score, best_iter = train_and_eval(train_df, val_df)
    print(f"  RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": f"cumulative_{i}", "groups": label, "rmsle": score})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))

  RMSLE = 2.81283  (best iteration: 36)

Ranking by RMSLE (lower = better)
      experiment                                                                                 groups    rmsle
        velocity                                                                               velocity 2.664201
           trend                                                                                  trend 2.719418
    cumulative_4                                        base + all_history + rolling_windows + velocity 2.731808
 rolling_windows                                                                        rolling_windows 2.735386
    cumulative_3                                                   base + all_history + rolling_windows 2.751903
    cumulative_7       base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 2.792482
    cumulative_5                                base + all_history + rolling_windows + velocity + trend 2.794928
    cumulative_6     

In [2]:
results = []

for i in range(2, len(FEATURES_GROUP)):
    if FEATURES_GROUP[i] != "velocity":
        subset = [FEATURES_GROUP[i], "velocity"]
        label = " + ".join(subset)
        train_df = merge_splits("split1", subset)
        val_df = merge_splits("split2", subset)
        score, best_iter = train_and_eval(train_df, val_df)
        results.append({"experiment": f"cumulative_{i}", "groups": label, "rmsle": score})

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


Ranking by RMSLE (lower = better)
  experiment                      groups    rmsle
cumulative_5              gmv + velocity 2.652512
cumulative_2  rolling_windows + velocity 2.664201
cumulative_6 conversion_rates + velocity 2.676128
cumulative_7              ewm + velocity 2.696676
cumulative_4            trend + velocity 2.703949
